# Backend-5 (추가) — Mastersample All OK 여부 (단발 실행용)

사양 요약  
- DB: `d1_machine_log.Main_machine_log`  
- 컬럼: `end_day`(text, YYYYMMDD), `end_time`(text, HH:MI:SS.xx), `contents`  
- 주간(day): `[D] 08:30:00 ~ 20:29:59`  
- 야간(night): `[D] 20:30:00 ~ 23:59:59` + `[D+1] 00:00:00 ~ 08:29:59`  
- 판정: window 내 `contents`에 `"Mastersample All OK"`가 1건이라도 있으면 **O**, 없으면 **X**  
- 추가 컬럼: **최초 발생 시각**(해당 window 내 최초 발생 end_time, 반올림 정규화 후)  
- 출력: day / night 각각 dataframe 출력 (저장/업서트 없음)

> ⚠️ 보안: 아래 DB 비밀번호는 기본값으로만 둬두고, 가능하면 환경변수로 주입하세요.


In [1]:
# =========================
# 0) 설치(필요 시)
# =========================
# !pip -q install sqlalchemy psycopg2-binary pandas


In [5]:
# =========================
# 1) Imports & Config  (FIXED: table name pinned)
# =========================
from __future__ import annotations

import os
from dataclasses import dataclass
from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

# DB 설정 (사양서)
DB_CONFIG = {
    "host": os.getenv("PG_HOST", "100.105.75.47"),
    "port": int(os.getenv("PG_PORT", "5432")),
    "dbname": os.getenv("PG_DBNAME", "postgres"),
    "user": os.getenv("PG_USER", "postgres"),
    # 기본값은 사양서 값. 가능하면 환경변수 PG_PASSWORD로 주입 권장.
    "password": os.getenv("PG_PASSWORD", ""),
}

# ✅ 너가 DBeaver에서 확인한 테이블로 "고정"
# - 스샷에 나온 것처럼 schema=d1_machine_log, table="Main_machine_log"
SCHEMA = "d1_machine_log"
TABLE  = "Main_machine_log"

# 이번 노트북은 '한 날짜'만 계산
PROD_DAY = "20260130"  # <-- 여기만 바꿔서 실행

def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    # 단발 실행 + 연결 1개 고정(풀 최소화)
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

engine = make_engine()

# ✅ 안전한 FQN(항상 따옴표로 감싸서 대소문자 이슈 제거)
FQN = f'"{SCHEMA}"."{TABLE}"'

# ✅ 존재 확인(없으면 여기서 바로 에러 내서 원인 명확히)
with engine.begin() as conn:
    ok = conn.execute(
        text("""
SELECT 1
FROM information_schema.tables
WHERE table_schema = :schema
  AND table_name = :table
"""),
        {"schema": SCHEMA, "table": TABLE},
    ).first()

if not ok:
    raise RuntimeError(f"테이블이 없습니다: {FQN} (schema={SCHEMA}, table={TABLE})")

print("engine ready | FQN =", FQN)

engine ready | FQN = "d1_machine_log"."Main_machine_log"


## 2) 시간 정규화(반올림) & window 조건

`end_time`은 text로 `"HH:MI:SS.xx"` 형태입니다.  
요구사항: `"HH:MI:SS"`로 **반올림(>=0.5초면 +1초)** 정규화 후, 문자열 비교로 window를 필터합니다.

아래 SQL은 다음을 수행합니다:
- `base_ts`: `end_day` + `substring(end_time,1,8)` 으로 초 단위 timestamp 생성
- `frac_num`: `substring(end_time from 9)` (예: `.88`)을 numeric으로 변환(없으면 0)
- `rounded_ts`: `frac_num>=0.5`면 1초 더함
- `rounded_time_text`: `HH24:MI:SS` 문자열


In [8]:
# =========================
# 3) Query builders
# =========================
@dataclass(frozen=True)
class WindowSpec:
    prod_day: str
    shift_type: str  # 'day' | 'night'
    next_day: str

def _next_day_yyyymmdd(prod_day: str) -> str:
    d = date(int(prod_day[0:4]), int(prod_day[4:6]), int(prod_day[6:8]))
    d2 = d + timedelta(days=1)
    return f"{d2.year:04d}{d2.month:02d}{d2.day:02d}"

def build_mastersample_sql(shift_type: str) -> text:
    """window 내 Mastersample All OK 존재 여부 + 최초 발생 시각(rounded)"""

    normalize_cte = f"""
WITH norm AS (
  SELECT
    end_day,
    end_time,
    contents,
    to_timestamp(end_day || ' ' || substring(end_time from 1 for 8), 'YYYYMMDD HH24:MI:SS')::timestamp AS base_ts,
    COALESCE(NULLIF(substring(end_time from 9), ''), '.0')::numeric AS frac_num
  FROM {FQN}
),
norm2 AS (
  SELECT
    end_day,
    end_time,
    contents,
    (base_ts + CASE WHEN frac_num >= 0.5 THEN interval '1 second' ELSE interval '0 second' END) AS rounded_ts,
    to_char((base_ts + CASE WHEN frac_num >= 0.5 THEN interval '1 second' ELSE interval '0 second' END)::time, 'HH24:MI:SS') AS rounded_time_text
  FROM norm
)
"""

    # ✅ 핵심: WHERE를 문자열로 박아넣지 말고, window 조건을 "표현식"으로 만든다
    if shift_type == "day":
        window_expr = """
( end_day = :prod_day
  AND rounded_time_text >= '08:30:00'
  AND rounded_time_text <= '20:29:59'
)
"""
    elif shift_type == "night":
        window_expr = """
(
    ( end_day = :prod_day AND rounded_time_text >= '20:30:00' )
 OR ( end_day = :next_day AND rounded_time_text <= '08:29:59' )
)
"""
    else:
        raise ValueError("shift_type must be 'day' or 'night'")

    sql = normalize_cte + f"""
, hits AS (
  SELECT rounded_ts
  FROM norm2
  WHERE {window_expr}
    AND contents LIKE '%Mastersample All OK%'
)
SELECT
  CASE WHEN EXISTS (SELECT 1 FROM hits) THEN 'O' ELSE 'X' END AS mastersample,
  (SELECT MIN(rounded_ts) FROM hits) AS first_ts
"""
    return text(sql)

In [9]:
# =========================
# 4) Run (day / night)
# =========================
df_day = fetch_mastersample(PROD_DAY, "day")
df_night = fetch_mastersample(PROD_DAY, "night")

print("=== DAY ===")
display(df_day)

print("=== NIGHT ===")
display(df_night)


=== DAY ===


,prod_day,shift_type,Mastersample,최초 발생 시각,updated_at
0,20260130,day,X,None,2026-01-31 15:11:04.268729+09:00


=== NIGHT ===


,prod_day,shift_type,Mastersample,최초 발생 시각,updated_at
0,20260130,night,X,None,2026-01-31 15:11:04.821190+09:00


## 5) 참고: 나중에 데몬/증분 스캔으로 확장하려면
- `hits`를 window 전체가 아니라 “마지막 처리 PK 이후”로 제한하는 조건을 추가하고,
- 결과 DF를 누적/업서트 저장할 테이블을 별도 설계하면 됩니다.
